# Week 5 — Build the evaluation test set (v3)

**Why v3.** The v2 run worked mechanically but produced a weak test set: all 913 questions came
from MedQuAD's MedlinePlus subset, which only carries `qtype=information`. That gave exactly two
question shapes ("What is (are) X ?" and "Do you have information about X"), median 6 words, and
nothing conversational. Two problems with that:

1. it never exercises the failure I actually found in week 3 (long conversational queries drifting
   off target), and
2. the topic title sits verbatim in the question, so retrieval looks artificially easy and hit
   rate@k won't discriminate between k values.

**Fix:** pull from MedQuAD's other subsets too. NIDDK, NINDS, CancerGov, CDC and SeniorHealth carry
the full question-type range — symptoms, causes, treatment, exams and tests, prevention, frequency —
phrased as "What are the symptoms of X ?", "What causes X ?", "How to diagnose X ?". Same parser,
more directories.

Three other things v3 handles that v2 didn't need to:
- **synonym matching** — those subsets use clinical names (Influenza) where MedlinePlus uses lay
  ones (Flu). The corpus `also_called` field bridges that; I checked, it recovers real misses.
- **duplicate questions** — NIDDK repeats identical question text across QAPairs (Acromegaly has
  "What are the treatments for..." three times). Deduped on (topic, question type).
- **question-type balance** — sampling now spreads across question types within each category,
  otherwise the 913 `information` questions would swamp everything.

Setup (once): `git clone https://github.com/abachaa/MedQuAD.git`

In [1]:
import json
import random
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path
import re

In [2]:
MEDQUAD_ROOT = Path('MedQuAD')

# subsets that still have their questions (3 MedlinePlus subsets had answers stripped for
# copyright, but we only need questions, so the Health Topics one is still usable)
SUBSETS = [
    '4_MPlus_Health_Topics_QA',   # information only, but maps 1:1 to my corpus
    '5_NIDDK_QA',                 # endocrine / digestive / kidney - full qtype range
    '6_NINDS_QA',                 # neurological - full qtype range
    '1_CancerGov_QA',             # cancers - full qtype range
    '9_CDC_QA',                   # infectious disease
    '7_SeniorHealth_QA',          # older-adult topics
]

CORPUS_JSON = 'topics_english_all.json'

TARGET_SIZE = 200
DEV_FRACTION = 0.5
MAX_PER_TOPIC = 3      # since a topic can now offer several question TYPES
ANALYSIS_MIN = 8       # categories at/above this can carry per-category claims
random.seed(42)

In [3]:
with open(CORPUS_JSON, encoding='utf-8') as f:
    corpus = json.load(f)

demographic_groups = {'Older Adults', 'Women', 'Men', 'Children and Teenagers', 'Population Groups'}

def primary_group(topic):
    for g in topic['groups']:
        if g not in demographic_groups:
            return g
    return topic['groups'][0] if topic['groups'] else 'Ungrouped'

def slug(url):
    return url.rstrip('/').split('/')[-1].lower()

by_title = {t['title'].lower(): t for t in corpus}
by_slug = {slug(t['url']): t for t in corpus}

# v3: synonym lookup. the non-MedlinePlus subsets use clinical names where MedlinePlus uses
# lay ones - e.g. Influenza vs Flu, Hearing Loss vs Hearing Disorders and Deafness.
# first topic wins if two topics share a synonym; rare and not worth resolving.
by_synonym = {}
for t in corpus:
    for s in t.get('also_called', []):
        by_synonym.setdefault(s.lower(), t)

print(f'corpus topics: {len(corpus)} | title keys: {len(by_title)} | synonym keys: {len(by_synonym)}')

corpus topics: 1016 | title keys: 1016 | synonym keys: 912


## Parse all subsets

Matching order: exact title, then synonym, then url slug (slug only for medlineplus.gov URLs —
the other sources have URLs like `.../acromegaly/Pages/fact-sheet.aspx` where the slug is
meaningless and could only produce false matches).

In [4]:
def match_topic(focus, url):
    f = focus.lower()
    if f in by_title:
        return by_title[f], 'title'
    if f in by_synonym:
        return by_synonym[f], 'synonym'
    if 'medlineplus' in url.lower():
        t = by_slug.get(slug(url))
        if t:
            return t, 'slug'
    return None, None


raw, unmatched, match_how = [], [], Counter()
per_subset = Counter()

for subset in SUBSETS:
    d = MEDQUAD_ROOT / subset
    if not d.exists():
        print(f'  !! missing directory, skipping: {d}')
        continue
    for path in sorted(d.glob('*.xml')):
        try:
            doc = ET.parse(path).getroot()
        except ET.ParseError:
            continue   # empty file (answers stripped)

        focus = (doc.findtext('Focus') or '').strip()
        url = doc.get('url', '')
        topic, how = match_topic(focus, url)
        if topic is None:
            if focus:
                unmatched.append((subset, focus))
            continue
        match_how[how] += 1

        for qa in doc.findall('.//QAPair'):
            q_el = qa.find('Question')
            if q_el is None or not (q_el.text and q_el.text.strip()):
                continue
            raw.append({
                'question': q_el.text.strip(),
                'qtype': q_el.get('qtype', '').strip() or 'unspecified',
                'topic_id': topic['id'],
                'topic_title': topic['title'],   # retrieval ground truth for hit rate@k
                'group': primary_group(topic),
                'source': subset,
            })
            per_subset[subset] += 1

print(f'raw question rows: {len(raw)}')
print(f'documents matched by: {dict(match_how)}')
print(f'unmatched focuses: {len(set(f for _, f in unmatched))}\n')
print('questions per subset:')
for s in SUBSETS:
    print(f'  {per_subset[s]:5d}  {s}')

raw question rows: 2138
documents matched by: {'title': 1030, 'slug': 7, 'synonym': 47}
unmatched focuses: 539

questions per subset:
    932  4_MPlus_Health_Topics_QA
    175  5_NIDDK_QA
    220  6_NINDS_QA
    238  1_CancerGov_QA
     11  9_CDC_QA
    562  7_SeniorHealth_QA


In [5]:

def normalize_question(text):
    """SeniorHealth questions arrive as 'Who is at risk for Leukemia? ?' - collapse the
    doubled punctuation and tidy whitespace before deduping, so near-identical variants
    collapse together too."""
    t = re.sub(r'\s+', ' ', text).strip()
    t = re.sub(r'\?\s*\?+', '?', t)        # '? ?' -> '?'
    t = re.sub(r'\s+([?.,!])', r'\1', t)   # ' ?'  -> '?'
    return t


malformed = sum(1 for q in raw if q['question'].count('?') > 1)

# dedupe: NIDDK repeats the same question text across several QAPairs (different answers,
# same question). keep one row per (topic, question type, normalised question text).
seen, questions = set(), []
for q in raw:
    q['question'] = normalize_question(q['question'])
    key = (q['topic_id'], q['qtype'], q['question'].lower())
    if key in seen:
        continue
    seen.add(key)
    questions.append(q)

print(f'normalised {malformed} malformed questions (double punctuation)')
print(f'after dedupe: {len(questions)} (removed {len(raw) - len(questions)} duplicates)')
print(f'\nquestion types available ({len(set(q["qtype"] for q in questions))}):')
for qt, n in Counter(q['qtype'] for q in questions).most_common():
    print(f'  {n:5d}  {qt}')

normalised 101 malformed questions (double punctuation)
after dedupe: 1564 (removed 574 duplicates)

question types available (15):
    960  information
    123  treatment
     87  research
     78  outlook
     71  symptoms
     61  exams and tests
     46  susceptibility
     44  causes
     26  prevention
     22  stages
     21  considerations
     11  frequency
     10  complications
      3  inheritance
      1  genetic changes


## Manual review pass

Sample across question types so I'm reviewing the variety, not 30 copies of the same shape.
Anything unusable goes in DROP and I re-run from here.

In [6]:
DROP = set()   # paste exact question text to exclude
questions = [q for q in questions if q['question'] not in DROP]

by_qt_all = defaultdict(list)
for q in questions:
    by_qt_all[q['qtype']].append(q)

print('review sample - up to 3 per question type:\n')
for qt in sorted(by_qt_all, key=lambda t: -len(by_qt_all[t])):
    print(f'--- {qt} ({len(by_qt_all[qt])}) ---')
    for q in random.sample(by_qt_all[qt], min(3, len(by_qt_all[qt]))):
        print(f'   {q["question"]}   [{q["source"]}]')
    print()

review sample - up to 3 per question type:

--- information (960) ---
   What is (are) Pelvic Pain?   [4_MPlus_Health_Topics_QA]
   What is (are) Botulism?   [4_MPlus_Health_Topics_QA]
   Do you have information about Anabolic Steroids   [4_MPlus_Health_Topics_QA]

--- treatment (123) ---
   What are the treatments for Rectal Cancer?   [1_CancerGov_QA]
   What are the treatments for Chronic Pain?   [6_NINDS_QA]
   What are the treatments for Complex Regional Pain Syndrome?   [6_NINDS_QA]

--- research (87) ---
   what research (or clinical trials) is being done for Huntington's Disease?   [6_NINDS_QA]
   what research (or clinical trials) is being done for Creutzfeldt-Jakob Disease?   [6_NINDS_QA]
   what research (or clinical trials) is being done for Cerebral Palsy?   [6_NINDS_QA]

--- outlook (78) ---
   What is the outlook for Skin Cancer?   [1_CancerGov_QA]
   What is the outlook for Complex Regional Pain Syndrome?   [6_NINDS_QA]
   What is the outlook for Small Intestine Cancer? 

## Sample: stratified by category, spread across question types

Two levels. Per topic, keep at most MAX_PER_TOPIC questions chosen from *different* question types,
so a single topic doesn't contribute three near-identical questions. Then allocate proportionally
across categories, and within each category round-robin over question types — otherwise the 900-odd
`information` questions from the MedlinePlus subset would crowd out symptoms/treatment/causes.

In [7]:
# --- per topic: prefer distinct question types ---
by_topic = defaultdict(list)
for q in questions:
    by_topic[q['topic_id']].append(q)

capped = []
for tid, qs in by_topic.items():
    buckets = defaultdict(list)
    for q in qs:
        buckets[q['qtype']].append(q)
    types = list(buckets)
    random.shuffle(types)
    for qt in types[:MAX_PER_TOPIC]:
        capped.append(random.choice(buckets[qt]))

print(f'after per-topic cap: {len(capped)} questions from {len(by_topic)} topics')


def sample_spread_by_qtype(pool, n):
    """take n from pool, round-robin across question types for an even spread."""
    buckets = defaultdict(list)
    for q in pool:
        buckets[q['qtype']].append(q)
    for v in buckets.values():
        random.shuffle(v)
    order = sorted(buckets, key=lambda t: -len(buckets[t]))
    picked, i = [], 0
    while len(picked) < n and any(buckets[t] for t in order):
        t = order[i % len(order)]
        if buckets[t]:
            picked.append(buckets[t].pop())
        i += 1
    return picked


by_group = defaultdict(list)
for q in capped:
    by_group[q['group']].append(q)

total = len(capped)
sample = []
for g in sorted(by_group, key=lambda g: -len(by_group[g])):
    share = max(1, round(TARGET_SIZE * len(by_group[g]) / total))
    sample.extend(sample_spread_by_qtype(by_group[g], min(share, len(by_group[g]))))

# trim overshoot from the largest categories only, never from ones near the analysis threshold
overshoot = len(sample) - TARGET_SIZE
if overshoot > 0:
    counts = Counter(q['group'] for q in sample)
    removable = sorted([q for q in sample if counts[q['group']] > ANALYSIS_MIN],
                       key=lambda q: -counts[q['group']])
    for q in removable[:overshoot]:
        sample.remove(q)

print(f'sampled {len(sample)} across {len(set(q["group"] for q in sample))} categories')
print('\nquestion-type mix in the final sample:')
for qt, n in Counter(q['qtype'] for q in sample).most_common():
    print(f'  {n:4d}  ({n/len(sample):4.0%})  {qt}')

after per-topic cap: 1173 questions from 934 topics
sampled 200 across 41 categories

question-type mix in the final sample:
    90  ( 45%)  information
    23  ( 12%)  treatment
    17  (  8%)  symptoms
    13  (  6%)  outlook
    12  (  6%)  causes
    10  (  5%)  research
    10  (  5%)  exams and tests
     8  (  4%)  susceptibility
     8  (  4%)  prevention
     3  (  2%)  frequency
     2  (  1%)  stages
     2  (  1%)  complications
     2  (  1%)  considerations


In [8]:
# how templated is it now? this is the number v2 failed on (2 openings, 100% coverage)
openings = Counter(' '.join(q['question'].split()[:3]).lower() for q in sample)
top10 = sum(n for _, n in openings.most_common(10))
lens = sorted(len(q['question'].split()) for q in sample)

print(f'distinct 3-word openings: {len(openings)}')
print(f'top 10 openings cover {top10/len(sample):.0%} of questions (v2 was 100% with only 2)')
print(f'length in words: min {lens[0]}, median {lens[len(lens)//2]}, max {lens[-1]}')
print('\nmost common openings:')
for o, n in openings.most_common(10):
    print(f'  {n:4d}  {o}...')

distinct 3-word openings: 23
top 10 openings cover 94% of questions (v2 was 100% with only 2)
length in words: min 3, median 6, max 12

most common openings:
    51  what is (are)...
    43  what are the...
    39  do you have...
    14  what is the...
    10  what research (or...
    10  how to diagnose...
     8  how to prevent...
     7  who is at...
     3  how many people...
     2  what to do...


## Category tiers — fixed before any experiment runs

In [9]:
counts = Counter(q['group'] for q in sample)
analysis_categories = sorted([g for g, n in counts.items() if n >= ANALYSIS_MIN])
aggregate_only = sorted([g for g, n in counts.items() if n < ANALYSIS_MIN])

print(f'ANALYSIS categories ({len(analysis_categories)}) - per-category claims allowed:')
for g in analysis_categories:
    print(f'  {counts[g]:3d}  {g}')
print(f'\nAGGREGATE-ONLY ({len(aggregate_only)} categories, {sum(counts[g] for g in aggregate_only)} questions)')

with open('analysis_categories.json', 'w', encoding='utf-8') as f:
    json.dump({'analysis_min': ANALYSIS_MIN,
               'analysis_categories': analysis_categories,
               'aggregate_only': aggregate_only,
               'category_counts': dict(counts)}, f, indent=2)
print('\nsaved analysis_categories.json')

ANALYSIS categories (7) - per-category claims allowed:
   14  Blood, Heart and Circulation
   16  Bones, Joints and Muscles
   17  Brain and Nerves
   19  Cancers
   17  Digestive System
   18  Infections
    8  Mental Health and Behavior

AGGREGATE-ONLY (34 categories, 91 questions)

saved analysis_categories.json


## Dev / held-out split

Tuning happens on dev; held-out is opened once at the end. Every analysis category appears in both.

In [10]:

topics_by_group = defaultdict(lambda: defaultdict(list))
for q in sample:
    topics_by_group[q['group']][q['topic_id']].append(q)

dev, test = [], []
for g, tmap in topics_by_group.items():
    tids = list(tmap)
    random.shuffle(tids)

    if len(tids) == 1:                      # single-topic category -> dev, aggregate-only anyway
        dev.extend(tmap[tids[0]])
        continue

    target = sum(len(tmap[t]) for t in tids) * DEV_FRACTION
    d, t_ = [], []
    for tid in tids:
        (d if len(d) < target else t_).extend(tmap[tid])

    # make sure neither side ended up empty when the category has >= 2 topics
    if not t_:
        m = tids[-1]
        t_ = [q for q in d if q['topic_id'] == m]
        d = [q for q in d if q['topic_id'] != m]
    if not d:
        m = tids[0]
        d = [q for q in t_ if q['topic_id'] == m]
        t_ = [q for q in t_ if q['topic_id'] != m]

    dev.extend(d)
    test.extend(t_)

random.shuffle(dev)
random.shuffle(test)

with open('test_queries_dev.json', 'w', encoding='utf-8') as f:
    json.dump(dev, f, indent=2, ensure_ascii=False)
with open('test_queries_heldout.json', 'w', encoding='utf-8') as f:
    json.dump(test, f, indent=2, ensure_ascii=False)

# --- integrity checks ---
dev_topics = {q['topic_id'] for q in dev}
test_topics = {q['topic_id'] for q in test}
assert not (dev_topics & test_topics), 'LEAK: topic appears in both splits'
assert not ({q['question'] for q in dev} & {q['question'] for q in test}), 'LEAK: question in both splits'

dc, tc = Counter(q['group'] for q in dev), Counter(q['group'] for q in test)
print(f'dev: {len(dev)} questions / {len(dev_topics)} topics')
print(f'held-out: {len(test)} questions / {len(test_topics)} topics')
print('no topic or question overlap between splits - confirmed')
print('analysis categories missing from a split:',
      [g for g in analysis_categories if dc[g] == 0 or tc[g] == 0] or 'none - guarantee holds')

print('\nqtype mix (dev / test):')
dq, tq = Counter(q['qtype'] for q in dev), Counter(q['qtype'] for q in test)
for qt in sorted(set(dq) | set(tq), key=lambda t: -(dq[t] + tq[t])):
    print(f'  {dq[qt]:3d} / {tq[qt]:3d}   {qt}')

print('\ncoverage by analysis category (dev / test):')
for g in analysis_categories:
    print(f'  {dc[g]:3d} / {tc[g]:3d}   {g}')

dev: 115 questions / 93 topics
held-out: 85 questions / 69 topics
no topic or question overlap between splits - confirmed
analysis categories missing from a split: none - guarantee holds

qtype mix (dev / test):
   56 /  34   information
   13 /  10   treatment
    8 /   9   symptoms
    9 /   4   outlook
    5 /   7   causes
    7 /   3   research
    5 /   5   exams and tests
    3 /   5   prevention
    3 /   5   susceptibility
    3 /   0   frequency
    1 /   1   stages
    1 /   1   complications
    1 /   1   considerations

coverage by analysis category (dev / test):
    7 /   7   Blood, Heart and Circulation
    9 /   7   Bones, Joints and Muscles
    9 /   8   Brain and Nerves
   10 /   9   Cancers
   10 /   7   Digestive System
    9 /   9   Infections
    4 /   4   Mental Health and Behavior


## Notes

- **subsets and matching:** pulled 6 subsets, 2,138 question rows. MedlinePlus 932,
  SeniorHealth 562, CancerGov 238, NINDS 220, NIDDK 175, CDC 11. Matched to my corpus by
  title (1,030), synonym (47 - Influenza→Flu etc), and url slug (7). 539 focuses didn't
  match, mostly rare diseases that aren't MedlinePlus topics. Fine, we sample from what did.

- **cleaning:** 101 questions had doubled punctuation from SeniorHealth ("Who is at risk for
  Leukemia? ?"), normalised before dedupe. 574 duplicates removed - NIDDK repeats the same
  question text across QAPairs with different answers. Left with 1,564.

- **question types:** 15 types now vs 1 in v2. Final sample is information 45%, treatment 12%,
  symptoms 8%, then outlook/causes/research/exams around 5-6% each. Information is still the
  biggest chunk because the MedlinePlus subset only has that type, but 45% beats 100%.

- **templating:** 23 distinct openings, top 10 cover 94% (v2 was 2 openings covering 100%).
  Length 3-12 words, median 6.

- **analysis categories:** 7 made the >=8 cutoff - Cancers 19, Infections 18, Brain and Nerves
  17, Digestive 17, Bones/Joints 16, Blood/Heart 14, Mental Health 8. Other 34 categories
  (91 questions) are aggregate-only.

- **split:** by topic, not by question, so nothing leaks. 115 dev / 85 held-out (~57/43),
  93 and 69 topics. Zero topic and question overlap, asserted. Held-out has 7-9 per analysis
  category except Mental Health at 4 - so report confidence intervals, not bare means, and
  treat Mental Health carefully.

- **limitation:** these are still curated NIH questions, not what someone actually types.
  Median 6 words, max 12, all template-derived. So this set won't catch the week-3 failure
  (long conversational queries drifting off target), and since the topic name is usually in
  the question, hit rate@k will look better than it should. CHQA-web or HealthSearchQA would
  fix that if there's time.